## 模块 3：月度经营趋势与环比增长分析（时间序列 + 窗口函数）
### 业务需求

基于原始表，按月统计核心经营指标，计算环比增长率，分析业务趋势变化

### 思路分析

1. 基于销售订单表，用`STRFTIME`函数提取年月，实现按月分组

2. 用`LAG()`窗口函数获取上月营收，计算环比增长率

3. 过滤无效小额订单，确保统计结果准确

### 核心知识点

- SQLite 日期函数`STRFTIME`使用

- 窗口函数`LAG()`偏移量取值

- 环比增长率计算逻辑

- `WHERE`过滤异常数据

### SQL 代码

In [1]:
import pandas as pd
import sqlite3

# 1. 连接SQLite数据库
conn = sqlite3.connect("/kaggle/input/datasets/tclaim/retail-sales/retail_sales.db")

# 4. 封装通用SQL执行函数，自动打印表格结果
def query_sql(sql):
    return pd.read_sql(sql, conn)
    
print("数据导入完成，调用 query_sql('SQL语句') 即可执行查询")

数据导入完成，调用 query_sql('SQL语句') 即可执行查询


In [2]:
sql = '''
WITH monthly_trend AS (
    SELECT
        STRFTIME('%Y-%m', 销售日期) AS 统计月份,
        COUNT(DISTINCT 订单ID) AS 订单总数,
        SUM(销售数量 * "单价(元)") AS 总营收,
        ROUND(AVG(销售数量 * "单价(元)"), 2) AS 客单价
    FROM "销售订单表"
    WHERE 销售数量 * "单价(元)" >= 10
    GROUP BY 统计月份
)
SELECT
    统计月份,
    订单总数,
    总营收,
    客单价,
    ROUND((总营收 - LAG(总营收) OVER (ORDER BY 统计月份)) * 100.0 / LAG(总营收) OVER (ORDER BY 统计月份), 2) AS `营收环比增长率(%)`
FROM monthly_trend
ORDER BY 统计月份;
'''
query_sql(sql)

,统计月份,订单总数,总营收,客单价,营收环比增长率(%)
0,2025-01,83,2848991.5,34325.20,NaN
1,2025-02,80,1621606.0,20270.08,-43.08
2,2025-03,75,1267578.5,16901.05,-21.83
3,2025-04,91,2032977.5,22340.41,60.38
4,2025-05,81,2797433.5,34536.22,37.60
5,2025-06,89,2639783.0,29660.48,-5.64


### 结果说明

输出月度经营趋势数据，包含订单量、营收、客单价、环比增长率，可直接用于经营复盘、业绩趋势分析，识别业务增长高峰与低谷